In [2]:
import os
os.chdir("../")

In [3]:
%pwd

'c:\\Users\\nahom\\OneDrive - Università Politecnica delle Marche\\Desktop\\AI Engineering\\Projects\\Complete-Medical-Chatbot-with-LLMs'

In [5]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [6]:
#Extract text from pdf files

def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
        )
    documents = loader.load()
    return documents

In [7]:
extracted_data = load_pdf_files("data")

In [11]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """ 
    Given a list of document objects, return a new list of document objects 
    containing only 'source' in metadata and the original page_content.
    """

    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content = doc.page_content,
                metadata = {"source": src}
            )
        )
    return minimal_docs

In [12]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [18]:
# Split the document into smaller chunks

def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 20,
        length_function = len
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [19]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 5859


In [25]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the huggingface embedding model
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name = model_name,
    )
    return embeddings

embedding = download_embeddings()

C:\Users\nahom\AppData\Local\Temp\ipykernel_31252\194776059.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\nahom\miniconda3\envs\medibot\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nahom\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` envi

In [30]:
# Test embedding generation

test_embedding = embedding.embed_query(
    "What is diabetes?"
)

print(f"Embedding dimensions: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

Embedding dimensions: 384
First 5 values: [-0.026402652263641357, 0.10922826826572418, -0.09243844449520111, 0.10056495666503906, -0.05268425494432449]


In [35]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


In [ ]:
from dotenv import load_dotenv
from pinecone import Pinecone

load_dotenv()

pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=PINECONE_API_KEY)


In [44]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384, #Dimension of the embeddings
        metric = "cosine", #Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [47]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)


In [48]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [52]:
retrieved_docs = retriever.invoke("What is acne?")
retrieved_docs

[Document(id='48cc7793-ae76-4e6e-8332-3358cda62184', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='ff8c12c1-9e30-4d98-90b7-f9f3b6d00927', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='2917d43e-61ed-4747-8966-9b5ea5b765a2', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged wi

In [55]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o")

In [59]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [61]:
system_prompt = (
    """ 
    You are a Medical assistant for question-answering tasks.
    Use the following pieces of retrieved context to answer the question.
    If you dont know the answer, say that you dont know. use three sentences maximum
    and keep the answer concise.
    \n\n
    {context}
    """
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [63]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [66]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism"})
print(response["answer"])

Acromegaly and gigantism are disorders resulting from the excessive release of growth hormone from the pituitary gland. In gigantism, this overproduction occurs before bone growth plates have closed, leading to abnormal height increase, while in acromegaly it occurs after bone growth stops, resulting in increased growth of bones and soft tissues as well as various bodily disturbances. Both conditions are rare and may result in delayed diagnosis due to gradual symptom development.


In [68]:
response = rag_chain.invoke({"input": "what is the treatement for Ance"})
print(response["answer"])

Treatment for acne typically includes topical treatments, oral medications, and lifestyle changes to manage symptoms. Common topical treatments include retinoids, benzoyl peroxide, and salicylic acid to reduce inflammation and bacteria. For more severe cases, oral antibiotics or hormonal therapies may be prescribed by a healthcare professional.
